# Figure 4 — a close pair with a contaminating neighbour

**In the paper:** Fig. 4; Sect. 5.1 (the map ladder and Λ_c); Sect. 4.3 (Eqs. 18, 19); the numbers in Sect. 5.2.

A scene with more than one source near the target: the number of sources is decided on the sky, one
at a time, against the campaign threshold Λ_c, and the measurement is a three-source moving fit with
the primary free and a prior on the neighbour's position.

| step | task | calls | paper |
|---|---|---|---|
| 1 | the scan law, the windows' zero points and the scene | `scanlaw.load_scan_law`, `sim.random_zero_points`, `sim.Scene`, `noise.gaia_like` | Sect. 4.2, Eq. 16; Sect. 5.2 |
| 2 | the windows and the per-window reduction, up to three sources | `ins.render_window`, `sim.simulate_window`, `red.ladder_window`, `red.delivery_mask`, `red.sigma_eta`; in one call `sim.harvest_campaign` | Sect. 2.2; App. A.2 |
| 3 | how many sources: the map ladder, Δχ² > Λ_c | `thr.lrt_threshold`, `search.next_peak`, `search.lambda_rung`, `ins.project_al`; in one call `search.map_ladder`; `search.describe`, `bp.peak_consistency` | Sect. 5.1 |
| 4 | the likelihood and the prior on C's position | `joint.loglike_profiled` | Sect. 4.3, Eqs. 18, 19 |
| 5 | the fit | `opt.minimize_restarts`, `joint.loglike_profiled`, `joint.stacked_design`, `joint.joint_fisher`, `joint.joint_covariance`; in one call `joint.fit_joint` | Sect. 5.2 |
| 6 | B relative to the fitted primary: the numbers of Sect. 5.2 | `joint.polar_elements` | Sect. 5.2 |
| 7 | what each piece of the model buys | `joint.fit_joint` | — |
| 8 | the figure | `bp.backprojection_map`, `radon.faststack` | Fig. 4 |

Every scanlib call carries the name of its module (`dyn.wls`, `joint.fit_joint`); the first code cell lists the
modules.

**The scene** (Sect. 5.2). The toy instrument with a Gaia-like per-sample noise for a primary of
G = 16 and a quiet background; Fig. 2's 47 transits with their epochs. B, a companion at 75 mas
(𝒮 = 0.2) moving 8 mas/yr; C, a static neighbour at 250 mas (𝒮 = 0.3), with a catalogue position of
assumed error 5 mas. Every window has its own sampling phase, its zero point η₀. The noise draw was
chosen by a rule fixed before looking (the code's cell of the noise draw).

**Paper and code.** Λ_c = `MAP_BAR`; Δχ² of a rung = `Lam`; π(Θ) = `prior`; the primary = A.
Each one-call form (`sim.harvest_campaign`, `search.map_ladder`, `joint.fit_joint`) is asserted equal to the
stage written out.

**What it reproduces** (Sect. 5.2). One, two, three sources in 3, 19, 25 windows; 69 detections;
Λ_c = 19.1; C accepted first, then B, each far above Λ_c; after B too few lines are left for a third candidate. B at
75.0 ± 0.1 mas, 110.2° ± 0.2°, μ = (−2.73 ± 0.10, −7.47 ± 0.09) mas/yr; C at 250.07 ± 0.05 mas,
300.00° ± 0.02°; flux-ratio errors 8 × 10⁻⁴ and 3 × 10⁻⁴; the primary consistent with zero. The
caption: 69 lines, 30 through B (along its fitted track), 40 through C; faststack at B and C 0.26 and 0.27 of the primary's
value. About 30 s.

In [ ]:
# the imports: numpy, matplotlib, and the scanlib modules; every scanlib call below carries its module's name
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

from scanlib import instrument as ins      # the instrument: sampling, LSF, the along-scan projection (Eq. 9)
from scanlib import noise                  # the noise model of a window
from scanlib import scanlaw                # scan angles, epochs and parallax factors
from scanlib import simulation as sim      # synthetic scenes and their windows
from scanlib import reduction as red       # one window, or a campaign, reduced to detections
from scanlib import backprojection as bp   # the triangulation map and its peak
from scanlib import radon                  # the inverse Radon transform and faststack
from scanlib import joint                  # the joint fit of all windows, and its helpers
from scanlib import search                 # the search for several sources (the map ladder)
from scanlib import thresholds as thr      # the threshold of the search, Lambda_c
from scanlib import optimize as opt        # the optimizer of the joint fit

In [ ]:
# the instrument, defined once (Sect. 2.1): 18 samples of 59 mas along scan, a Gaussian LSF of FWHM 100 mas,
# and its per-window margins Delta_1, Delta_2 (App. A.2; derived in auxil/sim_margins_lambda_w)
FWHM_MAS, PITCH_MAS, N_SAMPLES = 100.0, 59.0, 18
inst = ins.Instrument.gaussian(fwhm_mas=FWHM_MAS, pitch_mas=PITCH_MAS, n_samples=N_SAMPLES, margins=ins.TOY_MARGINS)

SAVE_PDF = False                           # True: write pdf/fig4_contaminated_binary.pdf

In [ ]:
# --- the noise draw and the windows' sampling phases. The seed was chosen by a rule stated
# before looking: the first seed from 20260928 up whose blind chain accepts two sources and
# whose injected B lies in the middle half of the fitted 4-D region (Delta chi2 1.92-5.39).
SEED = 20260929
# the seed of the zero points' own stream, a list of two whole numbers: [the noise seed, the label 3 of this stream]
ZERO_POINT_SEED = [SEED, 3]

In [ ]:
# --- the noise: the Gaia-like per-sample model of Sect. 5.2 for a primary of G = 16 whose transit sums nine CCD
# crossings. In electrons, sigma^2 = signal + 9 (b + r^2); noise.gaia_like returns it in the toy's flux units,
# (signal + background)/gain + read noise^2, with the gain from the primary's electrons (noise.gaia_electrons)
PRIMARY_G = 16.0
BACKGROUND_E_PER_SAMPLE_PER_CCD = 10.0     # b: the quiet background
NOISE_MODEL = noise.gaia_like(PRIMARY_G, sim.PRIMARY_FLUX, background=BACKGROUND_E_PER_SAMPLE_PER_CCD)
N_E = noise.gaia_electrons(PRIMARY_G)                     # e- from the primary per transit
GAIN_E_PER_UNIT = NOISE_MODEL.gain                        # e- per toy flux unit
N_CCD_PER_TRANSIT = noise.GAIA_N_CCD                      # the AF CCDs summed into one window
READ_NOISE_E_PER_CCD = noise.GAIA_READ_NOISE_E_PER_CCD    # r
LADDER_K_MAX = 3                           # per-window cap: a crowded scene

In [ ]:
# --- the scene: B, a moving companion inside one FWHM; C, a static neighbour
RHO_B = 75.0                               # B's separation from the primary [mas]
PA_B = 110.0                               # B's position angle [deg, from north through east]
Q_B = 0.2                                  # B's flux ratio to the primary
RHO_C = 250.0                              # C's separation from the primary [mas]
PA_C = 300.0                               # C's position angle [deg, from north through east]
Q_C = 0.3                                  # C's flux ratio to the primary
PM_B_MAS_YR = 8.0                          # B's relative proper motion, along the pair's tangent
CAT_SIGMA_MAS = 5.0                        # ASSUMED catalogue error on C


def polar(rho_mas, pa_deg):
    """The sky offset of a point given by its separation and its position angle.

    Inputs
        rho_mas   the separation from the primary [mas]
        pa_deg    the position angle [deg], from north through east
    Returns an array of two numbers
        dra*      the offset to the east [mas]
        ddec      the offset to the north [mas]"""
    angle_rad = np.deg2rad(pa_deg)
    return np.array([rho_mas * np.sin(angle_rad), rho_mas * np.cos(angle_rad)])


# the true positions of B and C at the reference epoch, each (dra*, ddec) [mas]
POS_B = polar(RHO_B, PA_B)
POS_C = polar(RHO_C, PA_C)
# B's proper motion (mu_a*, mu_d) [mas/yr]: its size times the unit vector at right angles to the line from the
# primary to B. That line's direction is (sin PA, cos PA); the tangent is (cos PA, -sin PA)
tangent_B = np.array([np.cos(np.deg2rad(PA_B)), -np.sin(np.deg2rad(PA_B))])
PM_B = PM_B_MAS_YR * tangent_B
# sim.Scene: the three sources, one row per source: the primary first, then B, then C
scene_positions = np.array([[0.0, 0.0], POS_B, POS_C])                 # (dra*, ddec) at the reference epoch [mas]
scene_fluxes = np.array([sim.PRIMARY_FLUX, Q_B * sim.PRIMARY_FLUX, Q_C * sim.PRIMARY_FLUX])   # [toy flux units]
scene_motions = np.array([[0.0, 0.0], PM_B, [0.0, 0.0]])               # (mu_a*, mu_d) [mas/yr]; only B moves
SCENE = sim.Scene(scene_positions, scene_fluxes, pm=scene_motions)

In [ ]:
# --- panel (a): the scene through an ASSUMED good imager (illustration only; nothing of it
# enters the analysis): Gaussian PSF of 70 mas FWHM on 40 mas pixels, Poisson noise at 4e4 e-
# in the primary's peak pixel, read noise 3 e-, sky 3e-5 of the peak
SCENE_HALF_MAS = 320.0                     # the panels' half-width [mas]
MAP_STEP_MAS = 1.0                         # the map's step [mas]
IMAGE_FWHM_MAS = 70.0                      # the imager's PSF, FWHM [mas]
IMAGE_PIXEL_MAS = 40.0                     # the imager's pixel [mas]
IMAGE_PEAK_E = 4.0e4                       # electrons in the peak pixel of an isolated primary
IMAGE_READ_E = 3.0                         # the read noise per pixel [e-]
IMAGE_SKY_FRACTION = 3.0e-5                # the sky per pixel, as a fraction of the primary's peak pixel
IMAGE_SEED = 20260929                      # the seed of the imager's noise (panel (a) only)


def scene_image(half_mas, scene=SCENE, fwhm_mas=IMAGE_FWHM_MAS, pixel_mas=IMAGE_PIXEL_MAS,
                peak_e=IMAGE_PEAK_E, read_e=IMAGE_READ_E, sky_fraction=IMAGE_SKY_FRACTION, rng=None):
    """The scene at the reference epoch as the assumed imager records it. A stand-in for a picture: it is drawn in
    panel (a) and enters no fit.

    Inputs
        half_mas       the image's half-width [mas]
        scene          the true scene (positions [mas], fluxes); the motions are not used
        fwhm_mas       the FWHM of the imager's Gaussian PSF [mas]
        pixel_mas      the pixel size [mas]
        peak_e         the electrons in the peak pixel of an isolated primary
        read_e         the read noise per pixel [e-]
        sky_fraction   the sky per pixel, as a fraction of the primary's peak pixel
        rng            a random generator: the detector's noise is added; None: no noise
    Returns two things
        image    the image, in units of an isolated primary's peak pixel; rows run along ddec, columns along dra*
        extent   (left, right, bottom, top) of the image [mas], for plt.imshow"""
    # step 1, the pixel grid: the PSF's sigma [mas], the number of pixels along each axis, and the pixel centres
    psf_sigma = fwhm_mas / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    n_pixels = int(round(2.0 * half_mas / pixel_mas))
    centres = -half_mas + pixel_mas * (np.arange(n_pixels) + 0.5)          # pixel centres
    pixel_x, pixel_y = np.meshgrid(centres, centres)                       # dra* and ddec of every pixel [mas]

    # step 2, the image without noise: the sky level, plus one Gaussian per source. Each source's height is its
    # flux over the primary's (scene.fluxes[0]: the first flux is the primary's)
    image = np.full_like(pixel_x, sky_fraction)
    for (source_x, source_y), flux in zip(scene.positions, scene.fluxes):
        distance_squared = (pixel_x - source_x) ** 2 + (pixel_y - source_y) ** 2
        image += (flux / scene.fluxes[0]) * np.exp(-0.5 * distance_squared / psf_sigma ** 2)

    # step 3, the detector's noise, in electrons: a Poisson draw of the image, plus the Gaussian read noise
    if rng is not None:
        electrons = image * peak_e
        noisy = rng.poisson(electrons) + rng.normal(0.0, read_e, image.shape)
        image = noisy / peak_e
    extent = (-half_mas, half_mas, -half_mas, half_mas)
    return image, extent

In [ ]:
# a helper of panel (d): the boundary of an error ellipse
def confidence_ellipse(centre, cov, n_sigma, n_points=200):
    """The boundary of the region that holds a 2-D Gaussian's probability of 1, 2 or 3 sigma (68.3, 95.4, 99.7 %).

    Inputs
        centre     the centre of the ellipse: two numbers, in the units of the two parameters
        cov        the 2 x 2 covariance of the two parameters
        n_sigma    1, 2 or 3
        n_points   the number of points on the boundary
    Returns an array of n_points rows and two columns: the points of the boundary, (x, y)."""
    # levels: a table from the number of sigma to the Delta chi^2 of that 2-D region; levels[1] is 2.30
    levels = {1: 2.30, 2: 6.18, 3: 11.83}
    # the ellipse's axes: the eigenvalues are the variances along them, the eigenvectors (columns) their directions
    eigenvalues, eigenvectors = np.linalg.eigh(np.asarray(cov, dtype=float))
    # the unit circle, one row (cos, sin) per point
    angle = np.linspace(0.0, 2.0 * np.pi, n_points)
    circle = np.column_stack([np.cos(angle), np.sin(angle)])
    # the circle stretched along the axes by sqrt(Delta chi^2 x variance), turned onto the axes, moved to the centre
    stretched = np.sqrt(levels[n_sigma]) * (circle * np.sqrt(eigenvalues))
    return np.asarray(centre) + stretched @ eigenvectors.T

In [ ]:
# a helper of panel (b): the primary's own line in every window, the one the reduction does not deliver
def primary_lines(campaign, psi, k_max=LADDER_K_MAX):
    """The primary's line in every window: the source that the per-window ladder placed nearest the primary's
    place in the window, -eta0_j. It is the one source the reduction does not deliver.

    Inputs
        campaign   the campaign of sim.harvest_campaign (with return_windows=True): its windows, the noise sigma of
                   their samples, and their zero points eta0 [mas]
        psi        the scan angle of every window [rad]
        k_max      the per-window cap on the number of sources
    Returns three arrays, one entry per window in which the ladder fitted a source
        the window's scan angle [rad]
        the primary's along-scan position on the sky [mas]
        its Fisher error [mas]"""
    line_psi = []
    line_eta = []
    line_sigma = []
    # j counts the windows
    for j, (psi_j, samples, zero_point) in enumerate(zip(psi, campaign.windows, campaign.eta0)):
        sigma_samples = campaign.sigma_noise[j]
        # red.ladder_window returns four things: the fitted positions [mas, in the window's frame], their fluxes,
        # the number of sources and a diagnostics record; the last two are not used here
        eta, flux, _n_sources, _diagnostics = red.ladder_window(samples, inst, sigma_samples, k_max=k_max)
        if eta.size == 0:
            continue
        # eta + zero_point: each fitted source's position on the sky; the smallest in size is the primary's
        nearest = int(np.argmin(np.abs(eta + zero_point)))                  # nearest the primary's place
        sigma_eta = red.sigma_eta(eta, flux, inst, sigma_samples)
        line_psi.append(float(psi_j))
        line_eta.append(float(eta[nearest] + zero_point))
        line_sigma.append(float(sigma_eta[nearest]))
    return np.array(line_psi), np.array(line_eta), np.array(line_sigma)

## 1. The scan law, the zero points and the scene (Sect. 4.2, Eq. 16; Sect. 5.2)

The primary at the origin, B moving, C at rest; each window sees each source at its η at that epoch
(Eq. 16 without parallax). Window $j$'s sample $n$ lies at $\eta_{0j} + g_n$ on the sky, with the zero
point $\eta_{0j}$ drawn within half a sample: a source at η is seen at $\eta - \eta_{0j}$ in the window.

In [ ]:
# the scan law (47 transits with their epochs), the windows' zero points, and a summary of the scene
# scanlaw.load_scan_law: the scan angles psi [rad] and the epochs t [yr]
psi, t = scanlaw.load_scan_law(scanlaw.PLEIADES)
print(f"scan law: {psi.size} transits over {t.max() - t.min():.1f} yr")
# sim.random_zero_points: one zero point eta0_j per window, drawn within half a sample (the window's sampling phase);
# its own stream (rng_zero_points, seeded by ZERO_POINT_SEED), so the noise does not depend on it
rng_zero_points = np.random.default_rng(ZERO_POINT_SEED)
eta0 = sim.random_zero_points(rng_zero_points, psi.size, inst)
print(f"zero points from {eta0.min():+.1f} to {eta0.max():+.1f} mas (rms {eta0.std():.1f})")
print(f"B at {RHO_B:.0f} mas ({RHO_B / FWHM_MAS:.2f} FWHM), PA {PA_B:.0f} deg, flux ratio {Q_B}, "
      f"moving ({PM_B[0]:+.2f}, {PM_B[1]:+.2f}) mas/yr")
print(f"C at {RHO_C:.0f} mas, PA {PA_C:.0f} deg, flux ratio {Q_C}, at rest")

In [ ]:
# the noise per sample [toy flux units] at the primary's peak sample (554 flux units) and in the wings
PEAK_SAMPLE_FLUX = 554.0                   # the primary's flux in its peak sample [toy flux units]
# two flux levels: the peak sample, and a sample with no light (the wings)
flux_levels = np.array([PEAK_SAMPLE_FLUX, 0.0])
sigma_peak, sigma_wing = NOISE_MODEL.sigma(flux_levels)
print(f"noise model: primary G = {PRIMARY_G:.0f}, {N_E:.0f} e- per transit, gain {GAIN_E_PER_UNIT:.1f} e- per toy "
      f"flux unit; sigma {sigma_peak:.2f} at the primary's peak sample, {sigma_wing:.2f} in the wings (toy units)")
print(f"per transit: background {N_CCD_PER_TRANSIT * BACKGROUND_E_PER_SAMPLE_PER_CCD:.0f} e- per sample (the quiet "
      f"case), read noise {np.sqrt(N_CCD_PER_TRANSIT) * READ_NOISE_E_PER_CCD:.1f} e-")

## 2. The windows and the per-window reduction (Sect. 2.2; App. A.2)

As in Fig. 3, but a window may hold three sources, so the ladder may climb a second rung (Λ_w = 10.8).
Every source but the one nearest the primary's place in the window, $-\eta_{0j}$, is delivered,
**unlabelled**: the reduction does not know which lines are B's and which C's.

In [ ]:
# the per-window margins Lambda_w; then the 47 windows, each rendered without noise and then with one noise draw
# inst.margins holds one margin per rung; [:LADDER_K_MAX - 1] keeps one per rung above the first source: here
# both, Delta_1 (to two sources) and Delta_2 (to three)
margins = inst.margins[:LADDER_K_MAX - 1]  # Lambda_w: Delta_1, Delta_2

# one window per transit: every source at its eta at that epoch, seen at eta - eta0_j
# rng_noise: the generator of the windows' noise. Its seed is SEED alone; the windows draw from it one after the
# other, in the order of the transits
rng_noise = np.random.default_rng(SEED)
windows = []                               # the noisy windows: one array of N_SAMPLES fluxes per transit
sigma_windows = []                         # the noise sigma of every sample: one array per transit
# j counts the transits
for j in range(psi.size):
    eta_j = SCENE.along_scan(psi[j], t[j]) - eta0[j]
    # ins.render_window: the noise-free window (flux per sample); NOISE_MODEL.sigma: one noise sigma per sample
    # from it
    clean_window = ins.render_window(eta_j, SCENE.fluxes, inst)
    sigma_samples = NOISE_MODEL.sigma(clean_window)
    # sim.simulate_window: the same window plus one noise draw
    samples = sim.simulate_window(eta_j, SCENE.fluxes, inst, rng_noise, sigma_samples)
    windows.append(samples)
    sigma_windows.append(sigma_samples)

In [ ]:
# reduce each window alone (the per-window ladder, up to three sources); every fitted source but the one nearest
# the primary's place is a detection, placed on the sky
det_window_list = []                       # each detection's window: the number j of its transit
det_eta_list = []                          # each detection's along-scan position on the sky [mas]
det_sigma_list = []                        # each detection's Fisher error [mas]
n_sources_list = []                        # the number of sources the ladder selected, one entry per window
# j counts the windows
for j, samples in enumerate(windows):
    # red.ladder_window: the window's own fit, one source, then more while each passes its margin; it returns the
    # fitted positions [mas, in the window's frame], their fluxes, the number of sources and a diagnostics record
    # (not used here)
    eta, flux, n_sources, _diagnostics = red.ladder_window(samples, inst, sigma_windows[j], k_max=LADDER_K_MAX)
    n_sources_list.append(n_sources)
    keep = red.delivery_mask(eta, origin=-eta0[j])                 # all but the source nearest the primary's place
    if keep.any():
        sigma_eta = red.sigma_eta(eta, flux, inst, sigma_windows[j])   # red.sigma_eta: each position's Fisher error [mas]
        # one detection per delivered source; eta_i + eta0[j] moves it from the window's frame onto the sky
        for eta_i, sigma_i in zip(eta[keep], sigma_eta[keep]):
            det_window_list.append(j)
            det_eta_list.append(eta_i + eta0[j])
            det_sigma_list.append(sigma_i)

In [ ]:
# the detections as arrays: each one's window, scan angle (psi[det_window]: the angle of its window), position
# on the sky, epoch and error; an error that is not finite gets the package's fallback
n_sources_per_window = np.array(n_sources_list)
det_window = np.array(det_window_list)
det_psi = psi[det_window]
det_eta = np.array(det_eta_list)
det_tau = t[det_window]                                  # each detection's epoch: the search needs it (step 3)
det_sigma = np.array(det_sigma_list)
# a singular Fisher matrix: the fallback
det_sigma = np.where(np.isfinite(det_sigma), det_sigma, red.SIGMA_DET_FALLBACK_MAS)

print(f"margins                          Delta_1 {margins[0]:.2f}, Delta_2 {margins[1]:.2f}")
# (n_sources_per_window == 1).sum(): the number of windows in which the ladder selected one source
print(f"windows {psi.size}: one source {(n_sources_per_window == 1).sum()}, two {(n_sources_per_window == 2).sum()}, "
      f"three {(n_sources_per_window == 3).sum()}")
print(f"unlabelled detections            {det_eta.size} from {np.unique(det_window).size} windows")

In [ ]:
# the convenience: sim.harvest_campaign renders and reduces in one call, from the same seed
# a new generator with the windows' seed, so that the call draws the same noise as the loop above
rng_noise_one_call = np.random.default_rng(SEED)
campaign = sim.harvest_campaign(SCENE, (psi, t), rng_noise_one_call, inst, NOISE_MODEL, k_max=LADDER_K_MAX,
                                return_windows=True, eta0=eta0)
# same windows and the same per-sample sigma, bit for bit (each assert compares window by window)
assert all(np.array_equal(a, b) for a, b in zip(campaign.windows, windows))
assert all(np.array_equal(a, b) for a, b in zip(campaign.sigma_noise, sigma_windows))
# same detections: angles, positions and errors
for a, b in ((campaign.psi, det_psi), (campaign.tau, det_tau), (campaign.eta, det_eta), (campaign.sigma, det_sigma)):
    assert np.array_equal(a, b)
print("sim.harvest_campaign reproduces the windows and the detections exactly")

# the noise sigma of every window for the later fits: one (N,) array per window
sigma_noise = campaign.sigma_noise

In [ ]:
# Sect. 5.2: the per-window fits select one, two and three sources in 3, 19 and 25
# windows; 69 detections are left for the map
n_one = (n_sources_per_window == 1).sum()
n_two = (n_sources_per_window == 2).sum()
n_three = (n_sources_per_window == 3).sum()
assert (n_one, n_two, n_three) == (3, 19, 25)
assert det_eta.size == 69

## 3. How many sources: the map ladder (Sect. 5.1)

Sources are added one at a time, each rung on all windows:

1. **propose** — the highest peak of the map (Eq. 13) of the detections still in the pool, outside 30 mas
   of the primary, crossed by at least three lines;
2. **score** — fit the $K$- and $(K+1)$-source models to every window, all sources moving and the primary
   free, and take $\Delta\chi^2 \equiv \chi^2_K - \chi^2_{K+1}$;
3. **test** — accept if $\Delta\chi^2 > \Lambda_{\rm c}$, the level that noise alone exceeds in 1 % of
   campaigns for a candidate searched anywhere in the field (closed form from the scan law;
   `auxil/sim_lambda_c.ipynb` checks it against simulation);
4. **assign** — remove the lines through the accepted source from the pool; back to 1.

The ladder stops at the first rejection, or when fewer than three lines are left. A free primary leans
towards a source not yet in the model, so that source's Δχ² is smaller than with the primary pinned
(the loop prints the primary's shift at every rung).

**The cells write the rungs out, then call `search.map_ladder`.** The written-out loop is the ladder's skeleton,
with a simpler step 4: it removes, from the pool, the lines that pass within max(3σ, the peak's width) of
the source just accepted. `search.map_ladder` instead rebuilds the pool after every acceptance from all the
detections, against every accepted source in the latest joint fit, with the fit's own error added to each
line's (the module docstring of `scanlib.search`, "The pool"). Both accept C, then B; the cells assert that
they accept the same number of sources and that the sources returned are those of the last accepted fit,
which holds every source together.

**A moving source's lines.** In step 4 a line is compared with the source's position at the line's own epoch,
the fitted position plus the fitted motion times τ: the line is moved to the reference epoch with the fitted
motion (Eq. 17) before the comparison. B moves 8 mas/yr, so its lines from the ends of the campaign lie up to
about 20 mas from its reference-epoch position; compared with that position alone they would stay in the pool
and propose B again. With the motion, B's lines leave with B, fewer than three lines are left, and the search
ends without a third candidate. The search needs each detection's epoch for this (`det_tau`).

In [ ]:
# the threshold of the map ladder
# thr.lrt_threshold: Lambda_c, the closed-form level of 1 % false additions for this scan law and a moving search
MAP_BAR = thr.lrt_threshold(psi, inst, t, eta0=eta0, free_primary=True)
# the same level with the primary pinned at the origin, printed beside it for comparison; not used after this cell
bar_pinned = thr.lrt_threshold(psi, inst, t, eta0=eta0)
print(f"threshold {MAP_BAR:.1f} (1 % false additions, closed form from the scan law, primary free; "
      f"pinned {bar_pinned:.4f}, free {MAP_BAR:.4f})")

In [ ]:
# the map ladder written out rung by rung: 1. propose, 2. score, 3. test, 4. assign; it stops at the first
# rejection, or when fewer than three lines are left in the pool
pool = np.ones(det_eta.size, bool)                        # the detections not yet assigned to a source
accepted = []                              # the accepted sources' positions at the reference epoch, (dra*, ddec) [mas]
widths = []                                # the width of each accepted source's map peak [mas]
peaks = []                                 # each accepted source's map peak, (dra*, ddec) [mas]
last_fit = None                            # the joint fit of the last accepted rung
while pool.sum() >= search.GATE_MIN_DETECTIONS:
    rung = len(accepted)                   # the rung's number: the number of sources accepted before it
    # search.next_peak: the pooled map's highest peak outside the origin zone, its width, and which pooled lines are
    # consistent with it; None when the map has no peak
    found = search.next_peak(det_psi[pool], det_eta[pool], det_sigma[pool])            # 1. propose
    if found is None:
        print(f"rung {rung}: the pooled map has no peak left -> stop")
        break
    peak, width, consistent = found
    if consistent.sum() < search.GATE_MIN_DETECTIONS:
        print(f"rung {rung}: {consistent.sum()} consistent lines -> stop")
        break
    # search.lambda_rung: Delta chi^2 of adding the candidate to the accepted sources, all moving, and the joint
    # fit with it
    Lam, rung_fit = search.lambda_rung(windows, psi, inst, sigma_noise, accepted, peak, tau=t,   # 2. score:
                                       eta0=eta0, free_primary=True)                             # chi2_K - chi2_K+1
    if Lam > MAP_BAR:                                                           # 3. test
        verdict = "accepted"
    else:
        verdict = "stop"
    # theta holds one block of four (a0, d0, mu_a, mu_d) per source, the primary's first; [:2] keeps the primary's
    # (a0, d0) and omits its motion and the other sources' blocks
    rung_theta = np.asarray(rung_fit["theta"])
    shift = rung_theta[:2]                               # the free primary's offset in the fit with the candidate
    print(f"rung {rung}: peak ({peak[0]:.0f}, {peak[1]:.0f}) mas, {consistent.sum()} of "
          f"{pool.sum()} lines consistent; Lambda {Lam:.1f} vs {MAP_BAR:.1f} -> {verdict}; "
          f"primary at ({shift[0]:+.2f}, {shift[1]:+.2f}) mas")
    if verdict == "stop":
        break
    # [-4:] keeps the last four entries of theta, the candidate's block; the blocks before it (the primary's, and
    # those of the sources accepted earlier) are omitted
    block = rung_theta[-4:]                              # the candidate's block (a0, d0, mu_a, mu_d)
    # [:2] keeps (a0, d0) and omits the motion (mu_a, mu_d)
    position = block[:2]                                 # its position at the reference epoch
    accepted.append(position)
    last_fit = rung_fit                                  # every source accepted so far is in this fit
    widths.append(width)
    peaks.append(peak)
    # 4. assign: a line belongs to the source when it passes within max(3 sigma, width) of the source's position at
    # the line's own epoch, position + motion x tau (Eq. 17: the line moved to the reference epoch with the fitted
    # motion). ins.project_al: Eq. 9. Simpler than search.map_ladder's pool rule, which also adds the fit's own error.
    dra_at_epoch = block[0] + block[2] * det_tau[pool]   # the source's dra* at each pooled line's epoch [mas]
    ddec_at_epoch = block[1] + block[3] * det_tau[pool]  # the source's ddec at each pooled line's epoch [mas]
    track = ins.project_al(dra_at_epoch, ddec_at_epoch, det_psi[pool])
    assigned = np.abs(det_eta[pool] - track) <= np.maximum(3.0 * det_sigma[pool], width)
    # np.flatnonzero(pool): the pooled lines' indices among all the detections; [assigned] keeps those of the
    # lines just assigned, and these leave the pool
    pool[np.flatnonzero(pool)[assigned]] = False
print(f"lines left in the pool: {pool.sum()} (fewer than {search.GATE_MIN_DETECTIONS}: no further candidate)")

In [ ]:
# search.map_ladder: the whole search in one call (propose, score, test, and the pool rebuilt after each acceptance)
ladder = search.map_ladder(campaign.psi, campaign.eta, campaign.sigma, campaign.windows, psi, inst,
                           campaign.sigma_noise, tau=t, det_tau=campaign.tau, bar=MAP_BAR, eta0=campaign.eta0,
                           free_primary=True)
assert ladder.n_sources == len(accepted) == 2, "the figure's scene must come out as two sources"
# the sources returned: the last accepted fit, which fits them all together (the primary's block first)
# reshape(-1, 4): one row (a0, d0, mu_a, mu_d) per source; [1:] omits the first row, the primary's
blocks = np.asarray(last_fit["theta"]).reshape(-1, 4)[1:]
# blocks[:, :2]: every row and the first two columns, the positions without the motions; ["flux"][1:]: every
# flux but the first, the primary's
assert np.array_equal(ladder.positions, blocks[:, :2])
assert np.array_equal(ladder.fluxes, np.asarray(last_fit["flux"])[1:])
print("search.map_ladder accepts the same two sources and returns them from the same fit")
print(search.describe(ladder))   # search.describe: one line per rung

In [ ]:
# truth: the two accepted sources are labelled B and C by B's true position, for presentation only
# the distance of each accepted source from B's true position [mas]
distance_from_B = []
for accepted_position in ladder.positions:
    distance_from_B.append(np.hypot(*(accepted_position - POS_B)))
# i_B, i_C: the numbers (0 or 1) of B and of C among the ladder's two sources, in the order of acceptance
i_B = int(np.argmin(distance_from_B))                                     # labelling uses the truth:
i_C = 1 - i_B                                                             # presentation only
# which the ladder accepted first
if i_C == 0:
    first_found = "C"
else:
    first_found = "B"
found_B = ladder.positions[i_B]
found_C = ladder.positions[i_C]
print(f"blind: B at ({found_B[0]:.1f}, {found_B[1]:.1f}) [truth ({POS_B[0]:.1f}, {POS_B[1]:.1f})], "
      f"C at ({found_C[0]:.1f}, {found_C[1]:.1f}) [truth ({POS_C[0]:.1f}, {POS_C[1]:.1f})]")

In [ ]:
# B's lines are counted along its fitted track (its block of the last fit), C's at its position (C is static)
block_B = blocks[i_B]                      # B's fitted block (a0, d0, mu_a, mu_d)
# B's along-scan position at each detection's epoch and scan angle: position + motion x tau, projected (Eq. 9)
track_B = ins.project_al(block_B[0] + block_B[2] * campaign.tau, block_B[1] + block_B[3] * campaign.tau,
                         campaign.psi)
# a line passes through B when it lies within max(3 sigma, the width of B's map peak) of the track
lines_B = np.abs(campaign.eta - track_B) <= np.maximum(3.0 * campaign.sigma, ladder.log[i_B]["width"])
# bp.peak_consistency returns (mask, distances); [0] keeps the mask of the lines through C, omits the distances
lines_C = bp.peak_consistency(campaign.psi, campaign.eta, campaign.sigma, found_C, ladder.log[i_C]["width"])[0]
print(f"lines through B: {lines_B.sum()}, through C: {lines_C.sum()}, "
      f"through neither: {(~lines_B & ~lines_C).sum()}")

In [ ]:
# Sect. 5.2: Lambda_c = 19.1; C accepted first, then B, each far above the bar;
# B's lines leave the pool with it, and too few are left for a third candidate
assert round(MAP_BAR, 1) == 19.1 and first_found == "C"
# ladder.log[:2]: the log's first two rungs, C's and B's; entry: one rung's record (a table), its Delta chi^2
# under "Lambda" and its verdict under "verdict"
assert all(entry["Lambda"] > 10 * MAP_BAR for entry in ladder.log[:2])
assert [entry["verdict"] for entry in ladder.log] == ["accepted", "accepted"] and pool.sum() < search.GATE_MIN_DETECTIONS

## 4. The likelihood and the prior (Sect. 4.3, Eqs. 18, 19)

Three sources, each with a block (Δα\*₀, Δδ₀, μα\*, μδ): twelve parameters. The primary is free: B lies
inside one FWHM, and a pinned primary would push any mis-centring into B. The fit maximizes the
posterior (Eq. 19),

$$\ln{\rm Pr}(\Theta, \vec f \mid \{\vec s_j\}) = \ln\mathcal L(\Theta, \vec f) + \ln\pi(\Theta),\qquad
-2\ln\pi(\Theta) = \sum_{i\in\{\Delta\alpha^*_C,\Delta\delta_C\}} \left(\frac{\Theta_i - m_i}{\sigma_i}\right)^2 + {\rm const},$$

with $\ln\mathcal L$ the profiled window likelihood of Eq. 18 and π a Gaussian on C's position only: its
catalogue position $m$ (drawn around the truth with the assumed 5 mas). The fluxes stay profiled and
free. The seed: the primary at rest at the origin, B at the map ladder's position, C at its catalogue
position, both at rest.

In [ ]:
# C's catalogue position: the truth plus an error drawn with the assumed sigma
# rng_catalogue: the generator of the catalogue error. Its seed is a list of two whole numbers: [the noise seed,
# the label 2 of this stream]. It gives two normal numbers: the errors in dra* and ddec [mas]
rng_catalogue = np.random.default_rng([SEED, 2])
catalogue_C = POS_C + rng_catalogue.normal(0.0, CAT_SIGMA_MAS, 2)
primary_seed = np.zeros(4)                 # the primary's block (a0, d0, mu_a, mu_d): at rest at the origin
# seed: primary at rest at the origin, B from the ladder, C from the catalogue
# np.r_ joins its entries into one array of twelve numbers, three blocks of (a0, d0, mu_a, mu_d)
seed_moving = np.r_[primary_seed, found_B, 0.0, 0.0, catalogue_C, 0.0, 0.0]
# the constrained entries of theta, their means (the seed's) and widths; the primary and B are free, C's motion too
# theta's entries: the primary's block 0-3, B's 4-7, C's 8-11, each (a0, d0, mu_a, mu_d); 8 and 9 are C's (a0, d0)
prior_index = np.r_[8, 9]                                                # C's position
prior_mean = seed_moving[prior_index]      # the prior's means: C's catalogue position [mas]
prior_sigma = np.array([CAT_SIGMA_MAS] * 2)   # the prior's widths: the catalogue error, once for dra*, once for ddec
# prior: the three arrays in the form joint.fit_joint and joint.joint_covariance take: (entries, means, widths)
prior = (prior_index, prior_mean, prior_sigma)


def prior_penalty(theta):
    """The prior's chi^2: -2 ln pi(theta) up to a constant (Eq. 19).

    Inputs
        theta   the twelve parameters: three blocks (a0, d0, mu_a, mu_d) [mas, mas, mas/yr, mas/yr]
    Returns one number: the sum over C's two position entries of ((theta - catalogue) / sigma)^2."""
    return np.sum(((theta[prior_index] - prior_mean) / prior_sigma) ** 2)

In [ ]:
# truth: C's catalogue position beside its true position; and the constrained entries with their sigma
print(f"catalogue position of C          ({catalogue_C[0]:.1f}, {catalogue_C[1]:.1f}) mas "
      f"[truth ({POS_C[0]:.1f}, {POS_C[1]:.1f})]")
print("constrained entries, sigma       ", prior_index, prior_sigma)

In [ ]:
# joint.loglike_profiled: ln L of Eq. 18 at theta, the fluxes at their best non-negative values (profiled out)
lnL_seed = joint.loglike_profiled(seed_moving, windows, psi, inst, sigma_noise, tau=t, eta0=eta0, free_primary=True)
print(f"at the seed: ln L {lnL_seed:.1f} nats, prior penalty {prior_penalty(seed_moving):.2f}")

## 5. The fit (Sect. 5.2)

The optimiser minimizes $-\ln{\rm Pr}$: minus the profiled log-likelihood (each sample whitened by its
own σ) plus half the prior's χ². Offsets are bounded to the window's reach, motions free. At the solution: the fluxes, the
Fisher matrix of the twelve parameters and three fluxes, and the covariance, the inverse of the Fisher
matrix plus the prior's curvature.

In [ ]:
# the measurement: minimise -ln Pr from the seed
def objective(theta):
    """The function the optimiser makes small: -ln Pr up to a constant (Eq. 19).

    Inputs
        theta   the twelve parameters: three blocks (a0, d0, mu_a, mu_d), the primary's first, then B's, then C's
    Returns one number: -ln L of the windows (Eq. 18, the fluxes profiled out) plus half the prior's chi^2."""
    lnL = joint.loglike_profiled(theta, windows, psi, inst, sigma_noise, tau=t, eta0=eta0, free_primary=True)
    return (-lnL
            + 0.5 * prior_penalty(theta))


# the bounds of one source's block: the two offsets within +-joint.THETA_BOUND, the two motions free; repeated for
# the three sources (block_bounds * 3)
block_bounds = [(-joint.THETA_BOUND, joint.THETA_BOUND)] * 2 + [(None, None)] * 2
all_bounds = block_bounds * 3
# opt.minimize_restarts: a bounded Nelder-Mead search from the seed, restarted from its answer until it stops
# improving. It returns the optimiser's result (result.x: the solution), the number of restarts, and whether it
# converged
result, n_restarts, converged = opt.minimize_restarts(
    objective, seed_moving, bounds=all_bounds,
    xatol=1e-4, maxiter_first=8000, maxiter_restart=32000)

In [ ]:
# at the solution: the fluxes, the chi2, the Fisher matrix, and the covariance with the prior's curvature
theta = np.asarray(result.x)               # the twelve fitted parameters
# with return_fluxes=True, joint.loglike_profiled returns two things: ln L (not used here) and the three fitted
# fluxes (the primary's, B's, C's)
_lnL, flux = joint.loglike_profiled(theta, windows, psi, inst, sigma_noise, tau=t, eta0=eta0, free_primary=True,
                                    return_fluxes=True)
# joint.stacked_design: every window's model columns, stacked; the model of all the samples is
# joint.stacked_design @ flux
design = joint.stacked_design(theta, psi, inst, tau=t, eta0=eta0, free_primary=True)
# np.ravel: all the windows' samples in one long array, in the order of the windows
residual = np.ravel(windows) - design @ flux
chi2 = float(np.sum((residual / np.ravel(sigma_noise)) ** 2))      # the windows' chi2, each sample whitened
# joint.joint_fisher: the Fisher matrix of theta and the fluxes
fisher = joint.joint_fisher(theta, flux, psi, inst, sigma_noise, tau=t, eta0=eta0, free_primary=True)
# joint.joint_covariance returns the covariance of theta and the fluxes, and the errors of theta
cov, sigma_theta = joint.joint_covariance(fisher, theta.size, prior)      # (F + prior curvature)^-1
# the number of samples of all the windows
n_samples_total = 0
for window in windows:
    n_samples_total += window.size
print(f"restarts {n_restarts}, converged {converged}; chi2 {chi2:.0f} "
      f"over {n_samples_total} samples; prior term {prior_penalty(theta):.2f}")

In [ ]:
# joint.fit_joint: the same fit in one call (seed, optimiser, fluxes, covariance)
fit_prior = joint.fit_joint(windows, psi, seed_moving, inst, sigma_noise, tau=t, prior=prior, free_primary=True,
                            eta0=eta0)
# joint.fit_joint minimizes the chi^2 of the windows plus the prior's, the cell above -ln L plus half the
# prior's: the same minimum, but the optimizer's stopping rule is relative to the objective's size, so
# the two stop a few 1e-5 of an error bar apart
assert np.all(np.abs(fit_prior["theta"] - theta) < 1e-3 * fit_prior["sigma_theta"])
assert np.allclose(fit_prior["cov"], cov, rtol=1e-3, atol=0.0)
print("joint.fit_joint reproduces the solution and its covariance within 1e-3 of each error bar")

## 6. B relative to the fitted primary (Sect. 5.2)

With a free primary, B's physics is its block minus the primary's: $\theta_{\rm rel} = \mathbf L\,\theta$,
covariance $\mathbf L\mathbf C\mathbf L^{\mathsf T}$, then in polar form (Eq. 12). This is panel (d) and
the numbers of Sect. 5.2.

In [ ]:
# B and C relative to the fitted primary: each one's block minus the primary's, with its covariance
def relative(theta, cov, k, n_par=4):
    """Source k's block minus the primary's, and the covariance of that difference.

    Inputs
        theta   the fitted parameters: one block (a0, d0, mu_a, mu_d) per source, the primary's first
        cov     the fit's covariance: the parameters first, then the fluxes
        k       the source's number in theta: 0 is the primary, 1 the next block, ...
        n_par   the number of parameters in a block
    Returns two things
        the relative block (dra*, ddec, mu_a*, mu_d) [mas, mas, mas/yr, mas/yr]
        its n_par x n_par covariance, L C L^T"""
    # L: the matrix of the text (theta_rel = L theta), n_par rows: +1 on source k's entries, -1 on the primary's
    L = np.zeros((n_par, theta.size))
    L[:, n_par * k:n_par * (k + 1)] = np.eye(n_par)       # the columns of source k's block
    L[:, :n_par] -= np.eye(n_par)                         # the first n_par columns: the primary's block
    # cov[:theta.size, :theta.size]: the parameters' rows and columns of the covariance; the fluxes' are omitted
    return L @ theta, L @ cov[:theta.size, :theta.size] @ L.T


# source 1 is B, source 2 is C (the order of the seed); source 0 is the primary
theta_B_rel, cov_B_rel = relative(fit_prior["theta"], fit_prior["cov"], 1)
theta_C_rel, cov_C_rel = relative(fit_prior["theta"], fit_prior["cov"], 2)

In [ ]:
# report: a relative block printed in polar form, with its errors
def report(name, theta_rel, cov_rel, extra=""):
    """Print a relative block in polar form with its errors.

    Inputs
        name        the label at the start of the printed line
        theta_rel   the block (dra*, ddec, mu_a*, mu_d) [mas, mas, mas/yr, mas/yr]
        cov_rel     its 4 x 4 covariance
        extra       text added at the end of the line
    Returns nothing; it prints one line."""
    # [:2] and [:2, :2]: the position (dra*, ddec) and its covariance. joint.polar_elements gives the separation rho
    # [mas], its error, the position angle PA [deg] and its error from them (Eq. 12)
    rho, sigma_rho, pa, sigma_pa = joint.polar_elements(theta_rel[:2], cov_rel[:2, :2])
    # [2:4] on the diagonal: the variances of the two proper motions
    sigma_mu = np.sqrt(np.diag(cov_rel)[2:4])
    print(f"{name:>16}: rho {rho:6.1f} +- {sigma_rho:.1f} mas, PA {pa:6.1f} +- {sigma_pa:.1f}, "
          f"mu ({theta_rel[2]:+.2f}, {theta_rel[3]:+.2f}) +- ({sigma_mu[0]:.2f}, {sigma_mu[1]:.2f}) mas/yr{extra}")

In [ ]:
# truth: B, C, the primary and the flux ratios as fitted, each beside its true value
report("B, the figure", theta_B_rel, cov_B_rel,
       f"   chi2 {fit_prior['chi2']:.0f}, chi2_prior {fit_prior['chi2_prior']:.2f}")
print(f"{'B, truth':>16}: rho {RHO_B:6.1f} mas, PA {PA_B:6.1f}, mu ({PM_B[0]:+.2f}, {PM_B[1]:+.2f}) mas/yr")
report("C", theta_C_rel, cov_C_rel)
# the primary's own fitted offset and residual motion (truth zero)
# [:4]: the first four entries of theta, the primary's block
primary_fit = fit_prior["theta"][:4]             # (a0, d0, mu_a, mu_d) of the primary
primary_sigma = fit_prior["sigma_theta"][:4]     # their errors
print(f"{'primary':>16}: offset ({primary_fit[0]:+.2f}, {primary_fit[1]:+.2f}) +- "
      f"({primary_sigma[0]:.2f}, {primary_sigma[1]:.2f}) mas, "
      f"residual motion ({primary_fit[2]:+.2f}, {primary_fit[3]:+.2f}) +- "
      f"({primary_sigma[2]:.2f}, {primary_sigma[3]:.2f}) mas/yr  [truth: zero]")
# flux ratios to the primary: flux[0] is the primary's, flux[1] B's, flux[2] C's
q_B_fit = fit_prior["flux"][1] / fit_prior["flux"][0]
q_C_fit = fit_prior["flux"][2] / fit_prior["flux"][0]
print(f"{'flux ratios':>16}: q_B {q_B_fit:.3f} (truth {Q_B}), q_C {q_C_fit:.3f} (truth {Q_C})")

In [ ]:
# Sect. 5.2: B at 75.0 +- 0.1 mas and 110.2 +- 0.2 deg, its proper motion (-2.73 +- 0.10,
# -7.47 +- 0.09) mas/yr; C at 250.07 +- 0.05 mas and 300.00 +- 0.02 deg; the flux ratios
# within their errors (8e-4, 3e-4) of the injected; the primary consistent with zero
# the fitted separation [mas], its error, the fitted position angle [deg] and its error, of B and of C (Eq. 12)
rho_B_fit, sigma_rho_B, pa_B_fit, sigma_pa_B = joint.polar_elements(theta_B_rel[:2], cov_B_rel[:2, :2])
rho_C_fit, sigma_rho_C, pa_C_fit, sigma_pa_C = joint.polar_elements(theta_C_rel[:2], cov_C_rel[:2, :2])
sigma_mu_B = np.sqrt(np.diag(cov_B_rel)[2:4])                 # the errors of B's two proper motions [mas/yr]
fitted_flux = fit_prior["flux"]                               # the three fitted fluxes: the primary's, B's, C's
# [12:, 12:]: the rows and columns after the twelve parameters
flux_cov = fit_prior["cov"][12:, 12:]                         # the fluxes' covariance block


def ratio_sigma(i):
    """The error of the flux ratio of source i to the primary, by linear propagation.

    Inputs
        i   the source's number among the fluxes: 1 is B, 2 is C
    Returns one number: the error of fitted_flux[i] / fitted_flux[0]."""
    # gradient: the derivative of the ratio with respect to each of the three fluxes
    gradient = np.zeros(3)
    gradient[0] = -fitted_flux[i] / fitted_flux[0] ** 2       # with respect to the primary's flux
    gradient[i] = 1.0 / fitted_flux[0]                        # with respect to the source's own flux
    return float(np.sqrt(gradient @ flux_cov @ gradient))


sigma_q_B = ratio_sigma(1)
sigma_q_C = ratio_sigma(2)
print(f"{'flux-ratio errors':>16}: q_B +- {sigma_q_B:.1e}, q_C +- {sigma_q_C:.1e}")
assert (round(rho_B_fit, 1), round(sigma_rho_B, 1), round(pa_B_fit, 1), round(sigma_pa_B, 1)) == (75.0, 0.1, 110.2, 0.2)
assert np.array_equal(np.round(theta_B_rel[2:4], 2), [-2.73, -7.47])
assert np.array_equal(np.round(sigma_mu_B, 2), [0.10, 0.09])
assert (round(rho_C_fit, 2), round(sigma_rho_C, 2), round(pa_C_fit, 2), round(sigma_pa_C, 2)) == (250.07, 0.05, 300.00, 0.02)
assert (round(sigma_q_B, 4), round(sigma_q_C, 4)) == (0.0008, 0.0003)
assert abs(q_B_fit - Q_B) < sigma_q_B and abs(q_C_fit - Q_C) < sigma_q_C
assert np.all(np.abs(primary_fit / primary_sigma) < 1.0), "the primary's offset and motion consistent with zero"
print("checks passed")

## 7. What each piece of the model buys

The same windows fitted with C ignored (B absorbs C's light and moves several σ), and with the
primary pinned at the window origin, against the figure's fit.

In [ ]:
# primary pinned at the window origin
# seed_moving[4:] omits the primary's block: theta is then (B, C), and C's position moves from entries 8, 9 to
# 4, 5 (prior_index - 4)
seed_pinned = seed_moving[4:]
prior_pinned = (prior_index - 4, prior_mean, prior_sigma)
fit_pinned = joint.fit_joint(windows, psi, seed_pinned, inst, sigma_noise, tau=t, prior=prior_pinned, eta0=eta0)
# C ignored: two sources, primary free, no priors
# the seed of two blocks: the primary at rest at the origin, B at the ladder's position and at rest
seed_ignore = np.r_[primary_seed, found_B, 0.0, 0.0]
fit_ignore = joint.fit_joint(windows, psi, seed_ignore, inst, sigma_noise, tau=t, free_primary=True, eta0=eta0)

# B from each fit, relative to the primary, against the figure's fit
theta_B_ignore, cov_B_ignore = relative(fit_ignore["theta"], fit_ignore["cov"], 1)
report("B, C ignored", theta_B_ignore, cov_B_ignore)
# with the primary pinned, B's block is the first four entries of theta ([:4]), measured from the pinned primary
report("B, primary pinned", fit_pinned["theta"][:4], fit_pinned["cov"][:4, :4])
report("B, primary free", theta_B_rel, cov_B_rel, "   <- the figure")

## 8. The figure (Fig. 4)

(a) The scene through a good imager, for illustration. (b) The pooled lines; C's darker, the primary's
dotted, B's map peak (circle) and truth (star). (c) faststack of the same windows (App. C.2), true
positions marked. (d) B relative to the primary: the joint 1, 2, 3σ regions (Δχ² = 2.30, 6.18, 11.83;
68.3, 95.4, 99.7 %) of each pair of its four parameters. Plotting only;
the cell after it prints the caption's numbers.

In [ ]:
# --- FAST cell: plotting only (the paper's grayscale style: identity by marker
# shape and fill -- B a filled circle, C a filled square, truth an open ring).
# Tweak and re-run this cell alone. ---
plt.rcParams.update({"font.size": 10.0, "axes.linewidth": 0.5,
                     "xtick.labelsize": 9.0, "ytick.labelsize": 9.0})
COL_HOST, COL_ACCENT = "k", "tomato"                          # tomato: the accent of Figs 1-3
# the marker vocabulary of Figs 2-3, kept here: truth = white star with a
# black edge; the primary / the window origin = black plus; the joint fit =
# black cross; the map's proposed peak (a seed, never a measurement) = open
# tomato circle
TRUTH = dict(marker="*", s=170, facecolor="white", edgecolor="k", lw=0.9)
FIT = dict(marker="x", s=70, color="k", lw=1.6)
SEED_MARK = dict(marker="o", s=60, facecolor="none", edgecolor=COL_ACCENT, lw=1.1)
# (a)-(c) are image panels; (d) is placed by hand beside them
fig = plt.figure(figsize=(17.6, 4.4))
outer = fig.add_gridspec(1, 4, wspace=0.30, width_ratios=[1.0, 1.0, 1.0, 1.0])
ax_a, ax_b, ax_c = (fig.add_subplot(outer[0, i]) for i in range(3))

# (a) the scene as a good imager records it
img_a, ext_a = scene_image(SCENE_HALF_MAS, rng=np.random.default_rng(IMAGE_SEED))
ax_a.imshow(np.sqrt(np.clip(img_a, 0.0, None)), origin="lower", extent=ext_a, cmap="Greys",
            vmin=0.0, vmax=1.0, interpolation="nearest", rasterized=True, zorder=1)
ax_a.add_patch(plt.Circle((SCENE_HALF_MAS - 45.0, -SCENE_HALF_MAS + 45.0), IMAGE_FWHM_MAS / 2.0,
                          facecolor="0.3", edgecolor="k", lw=0.6, zorder=6))
ax_a.annotate("B", xy=POS_B, xytext=(170, -120), textcoords="data", fontsize=9, weight="bold",
              ha="center", va="center",
              arrowprops=dict(arrowstyle="-|>", color="k", lw=0.8, shrinkB=6), zorder=6)
ax_a.annotate("C", xy=POS_C, xytext=(-110, 230), textcoords="data", fontsize=9, weight="bold",
              ha="center", va="center",
              arrowprops=dict(arrowstyle="-|>", color="k", lw=0.8, shrinkB=8), zorder=6)
ax_a.set_xlim(SCENE_HALF_MAS, -SCENE_HALF_MAS)
ax_a.set_ylim(-SCENE_HALF_MAS, SCENE_HALF_MAS)
ax_a.set_aspect("equal")
ax_a.set_xlabel(r"$\Delta\alpha^{*}$  [mas]   (East $\leftarrow$)")
ax_a.set_ylabel(r"$\Delta\delta$  [mas]")

# (b) the pooled lines over the whole scene; C's lines darker
# at this brightness the lines' own widths are tenths of a milliarcsecond,
# hairlines on a 1 mas grid: the DISPLAY draws every line with a floor of
# LINE_FLOOR_MAS (the analysis uses the true widths)
LINE_FLOOR_MAS = 4.0
sigma_show = np.maximum(campaign.sigma, LINE_FLOOR_MAS)
# bp.backprojection_map: the map of Eq. 13 on a grid, returned with its axis and its extent
sky, _axis, extent = bp.backprojection_map(campaign.psi, campaign.eta, sigma_show,
                                        half_size=SCENE_HALF_MAS, step=MAP_STEP_MAS)
# the same map from the lines through C
sky_C, _axis, _ext = bp.backprojection_map(campaign.psi[lines_C], campaign.eta[lines_C], sigma_show[lines_C],
                                        half_size=SCENE_HALF_MAS, step=MAP_STEP_MAS)
# Display only. One grey scale for every line, saturating at MAP_VMAX_FRACTION of the full
# map's peak (C's crossing), so the lines through B read dark enough to follow. The lines
# through C are drawn through the SAME scale but as a translucent overlay capped at
# C_OVERLAY_ALPHA: darker than B's single lines, not black, so their crossing does not
# swamp the panel. (The map is a sum of ridges, so the lines not through C are sky - sky_C.)
MAP_VMAX_FRACTION = 0.4
C_OVERLAY_ALPHA = 0.6
vmax_b = MAP_VMAX_FRACTION * sky.max()
ax_b.imshow(sky - sky_C, origin="lower", extent=extent, cmap="Greys", vmin=0.0, vmax=vmax_b,
            rasterized=True, zorder=1)
overlay = np.zeros(sky_C.shape + (4,))
# overlay[..., 3]: the alpha channel of the RGBA image; the three colour channels stay zero (black)
overlay[..., 3] = np.clip(sky_C / vmax_b, 0.0, 1.0) * C_OVERLAY_ALPHA
ax_b.imshow(overlay, origin="lower", extent=extent, rasterized=True, zorder=2)
# the primary's own lines (the component the reduction keeps back), pooled
# and drawn as contours: identity by line style, not by colour
# (the primary's own error bars are a fraction of a mas, so its lines are
# drawn with a display width floor of PRIMARY_LINE_FLOOR_MAS: contours of
# hair-thin ridges would be a star of hairlines)
PRIMARY_LINE_FLOOR_MAS = 6.0
# the primary's lines: their scan angles [rad], positions on the sky [mas] and errors [mas]
primary_psi, primary_eta, primary_sig = primary_lines(campaign, psi)
sky_P, axis_P, _ext = bp.backprojection_map(primary_psi, primary_eta, np.maximum(primary_sig, PRIMARY_LINE_FLOOR_MAS),
                                         half_size=SCENE_HALF_MAS, step=MAP_STEP_MAS)
ax_b.contour(axis_P, axis_P, sky_P, levels=np.array([0.15, 0.35, 0.6]) * sky_P.max(),
             colors="k", linewidths=0.7, linestyles="dotted", zorder=3)
# one FWHM around the primary, primary marker, map peak and truth
ax_b.add_patch(plt.Circle((0, 0), FWHM_MAS, fill=False, ls=(0, (4, 3)), ec="#666666", lw=1.0))
ax_b.plot(0, 0, "+", color=COL_HOST, ms=11, mew=1.6, zorder=6)
peak_B = ladder.log[i_B]["peak"]
ax_b.scatter([peak_B[0]], [peak_B[1]], zorder=6, **SEED_MARK)
ax_b.scatter([POS_B[0]], [POS_B[1]], zorder=7, **TRUTH)
first = "C" if i_C == 0 else "B"   # which source the ladder found first
ax_b.set_xlim(SCENE_HALF_MAS, -SCENE_HALF_MAS)
ax_b.set_ylim(-SCENE_HALF_MAS, SCENE_HALF_MAS)
ax_b.set_aspect("equal")
ax_b.set_xlabel(r"$\Delta\alpha^{*}$  [mas]")
ax_b.set_ylabel(r"$\Delta\delta$  [mas]")

# (c) the picture a stacking reconstruction makes of the same windows: faststack
# (Harrison 2011), the robust stack of the window samples that SEAPipe builds,
# over the whole frame. Each image pixel collects one sample from every window
# whose strip covers it; the lowest few set the level and higher samples (the
# primary's light in a strip that crossed it) are weighted down. No smoothing:
# the islands' edges are the sample strips at the law's scan angles. The zero
# level is white; the stretch is linear to 0.6 of the value at the primary.
# Marked: the primary's plus and the TRUE positions of B and C (stars); no
# position is read on this picture, the measurement is panel (d).
# radon.faststack returns the image first, then its variance, counts and axis; [0] keeps the image
faststack_image = radon.faststack(psi, windows, inst, half=SCENE_HALF_MAS, step=2.0, eta0=eta0)[0]
centre_pixel = faststack_image.shape[0] // 2
ax_c.imshow(faststack_image / faststack_image[centre_pixel, centre_pixel], origin="lower",
            extent=(-SCENE_HALF_MAS, SCENE_HALF_MAS, -SCENE_HALF_MAS, SCENE_HALF_MAS),
            cmap="gray_r", vmin=0.0, vmax=0.6, interpolation="nearest", rasterized=True, zorder=1)
# fitted primary offset (the plus) and the true positions
primary_position = fit_prior["theta"][:2]
# white stroke so marks read on the dark core
stroke = [pe.withStroke(linewidth=2.2, foreground="white")]
ax_c.plot(primary_position[0], primary_position[1], "+", color="white", ms=10, mew=1.6, zorder=8,
          path_effects=[pe.withStroke(linewidth=3.0, foreground="k")])
ax_c.scatter([POS_B[0], POS_C[0]], [POS_B[1], POS_C[1]], zorder=7, **TRUTH)
# labels, each about 22 mas clear of its marker
ax_c.text(primary_position[0] - 22, primary_position[1] + 18, "A", fontsize=8.5, weight="bold", ha="left", va="bottom",
          zorder=9, path_effects=stroke)
ax_c.text(POS_B[0] + 26, POS_B[1] - 22, "B", fontsize=8.5, weight="bold", ha="right", va="top",
          zorder=9, path_effects=stroke)
ax_c.text(POS_C[0] - 26, POS_C[1] + 22, "C", fontsize=8.5, weight="bold", ha="right", va="bottom",
          zorder=9, path_effects=stroke)
ax_c.set_xlim(SCENE_HALF_MAS, -SCENE_HALF_MAS)
ax_c.set_ylim(-SCENE_HALF_MAS, SCENE_HALF_MAS)
ax_c.set_aspect("equal")
ax_c.set_xlabel(r"$\Delta\alpha^{*}$  [mas]")
ax_c.set_ylabel(r"$\Delta\delta$  [mas]")

# (d) the corner grid of B relative to the primary: (dra*, ddec, mu_a*, mu_d)
labels = [r"$\Delta\alpha^{*}$ [mas]", r"$\Delta\delta$ [mas]",
          r"$\mu_{\alpha^*}$ [mas/yr]", r"$\mu_\delta$ [mas/yr]"]
truth_B = np.r_[POS_B, PM_B]                                   # B's true (dra*, ddec, mu_a*, mu_d)
sigma_B_rel = np.sqrt(np.diag(cov_B_rel))                              # the errors of B's four relative parameters
# the corner grid fills a box of exactly panel (c)'s size (the image panels
# are square and shrink inside their slots; the grid must follow them)
for ax in (ax_a, ax_b, ax_c):
    ax.apply_aspect()
box_c, box_b = ax_c.get_position(), ax_b.get_position()
gap = box_c.x0 - box_b.x1                                         # the gap the other panels have
box = dict(x0=box_c.x1 + gap, y0=box_c.y0, w=box_c.width, h=box_c.height)
pad = 0.045                                                       # between the small axes, in box units
cell_w = box["w"] * (1.0 - 3 * pad) / 4.0
cell_h = box["h"] * (1.0 - 3 * pad) / 4.0
def corner_axis(i, j):   # returns the small axes in row i, column j of the grid
    return fig.add_axes([box["x0"] + j * (cell_w + pad * box["w"]),
                         box["y0"] + (3 - i) * (cell_h + pad * box["h"]), cell_w, cell_h])
corner_axes = {}
# diagonal: Gaussian marginals; below it: 1, 2, 3 sigma ellipses
for i in range(4):
    for j in range(i + 1):
        ax = corner_axis(i, j)
        corner_axes[(i, j)] = ax
        lo_j, hi_j = theta_B_rel[j] - 3.5 * sigma_B_rel[j], theta_B_rel[j] + 3.5 * sigma_B_rel[j]
        if i == j:
            x = np.linspace(lo_j, hi_j, 200)
            ax.plot(x, np.exp(-0.5 * ((x - theta_B_rel[j]) / sigma_B_rel[j]) ** 2), "-", color="k", lw=1.0)
            ax.axvline(truth_B[j], color="k", lw=0.9, ls=(0, (3, 2)))
            ax.set_ylim(0.0, 1.15)
            ax.set_yticks([])
        else:
            lo_i, hi_i = theta_B_rel[i] - 3.5 * sigma_B_rel[i], theta_B_rel[i] + 3.5 * sigma_B_rel[i]
            # np.ix_([j, i], [j, i]): the 2x2 covariance of parameter j (the x axis) and parameter i (the y axis)
            sub_cov = cov_B_rel[np.ix_([j, i], [j, i])]
            for n_sig, shade in ((3, "0.85"), (2, "0.65"), (1, "0.40")):
                ring = confidence_ellipse([theta_B_rel[j], theta_B_rel[i]], sub_cov, n_sig)
                ax.fill(ring[:, 0], ring[:, 1], color=shade, lw=0, zorder=2)
            ax.scatter([truth_B[j]], [truth_B[i]], zorder=4, **{**TRUTH, "s": 90})
            ax.scatter([theta_B_rel[j]], [theta_B_rel[i]], zorder=5, **{**FIT, "s": 40, "lw": 1.2})
            ax.set_ylim(lo_i, hi_i)
        ax.set_xlim(lo_j, hi_j)
        if j in (0, 2):                     # sky and motion axes run east-positive to the LEFT
            ax.set_xlim(hi_j, lo_j)
        ax.tick_params(labelsize=6.2, length=2, pad=1)
        ax.locator_params(nbins=3)
        if i < 3:
            ax.set_xticklabels([])
        else:
            ax.set_xlabel(labels[j], fontsize=7.0, labelpad=1)
        if j > 0 or i == 0:
            ax.set_yticklabels([])
        else:
            ax.set_ylabel(labels[i], fontsize=7.0, labelpad=1)
corner_axes[(0, 0)].set_title("B, relative to the primary", fontsize=8.0, loc="left")
# panel letters
for ax, letter in ((ax_a, "(a)"), (ax_b, "(b)"), (ax_c, "(c)"), (corner_axes[(0, 0)], "(d)")):
    ax.text(-0.02 if ax is not corner_axes[(0, 0)] else -0.28, 1.04, letter, transform=ax.transAxes,
            ha="right", va="bottom", fontsize=10.5, weight="bold")

In [ ]:
# numbers for the caption
print(f"noise: Gaia-like per-sample model, primary G = {PRIMARY_G:.0f}, B and C fainter by "
      f"{-2.5 * np.log10(Q_B):.2f} and {-2.5 * np.log10(Q_C):.2f} mag (flux ratios {Q_B} and {Q_C}); sigma "
      f"{sigma_peak:.2f} at the primary's peak sample, {sigma_wing:.2f} in the wings (toy units), quiet background. "
      f"The imager's frame: {IMAGE_PEAK_E:.0e} e- in the peak pixel, "
      f"read noise {IMAGE_READ_E:.0f} e-, sky {IMAGE_SKY_FRACTION:.0e} of the peak per pixel (assumed).")

print(f"panel (b): {campaign.eta.size} unlabelled lines pooled; {lines_B.sum()} through B, "
      f"{lines_C.sum()} through C (darker); map ladder accepted {first} first")
# the pixel centres of panel (c)'s faststack image along each axis [mas]: the same half-width and the same 2 mas
# step as the call of radon.faststack in the figure's cell
faststack_axis = np.arange(-SCENE_HALF_MAS, SCENE_HALF_MAS + 1e-9, 2.0)


def faststack_at(position):
    """Faststack at the pixel nearest a sky position, over its value at the primary.

    Inputs
        position   the sky position (dra*, ddec) [mas]
    Returns one number: the image's value there over its value at the centre pixel (the origin)."""
    row = np.argmin(np.abs(faststack_axis - position[1]))         # the row nearest ddec
    column = np.argmin(np.abs(faststack_axis - position[0]))      # the column nearest dra*
    return faststack_image[row, column] / faststack_image[centre_pixel, centre_pixel]


print(f"panel (c): faststack at the true B {faststack_at(POS_B):.2f} and C {faststack_at(POS_C):.2f} of its value at the primary; zero level white, stretch to 0.6")

# the caption's numbers
assert (campaign.eta.size, int(lines_B.sum()), int(lines_C.sum()), first) == (69, 30, 40, "C")
assert (round(faststack_at(POS_B), 2), round(faststack_at(POS_C), 2)) == (0.26, 0.27)

In [ ]:
# write the PDF only when SAVE_PDF is True
if SAVE_PDF:
    Path("pdf").mkdir(exist_ok=True)
    fig.savefig(Path("pdf") / "fig4_contaminated_binary.pdf", bbox_inches="tight")
    print("wrote pdf/fig4_contaminated_binary.pdf")
else:
    print("SAVE_PDF is False: the PDF is not written")